# Train YOLO11n on ACDC — Efficient Improved Baseline

This notebook trains a **YOLO11n** model with efficiency-optimized hyperparameters:
- **512px** image size (same as baseline — 2.25x fewer pixels than 640px)
- **50 epochs** with cosine LR scheduler (most gain happens in first 50 epochs)
- **AdamW optimizer** with warmup
- **Stronger augmentation**: mosaic, mixup, copy-paste
- **No layer freezing** (full fine-tuning)
- **Per-class evaluation** after training

Expected improvement: **+3-7 mAP@0.5** over the YOLOv8n baseline (0.2664)
in roughly **1 hour** on a T4 GPU.

In [ ]:
# Cell 1: Setup — Mount Drive, clone repo, install dependencies
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Clone or pull the repo
REPO_DIR = Path('/content/ETH_AI_OBJ_DETECTION')
if REPO_DIR.exists():
    print('Repo exists, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/Danielit707/ETH_AI_OBJ_DETECTION.git', str(REPO_DIR)], check=True)

# Force upgrade ultralytics to latest (required for YOLO11)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ultralytics'], check=True)

from ultralytics import YOLO
import torch
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths — matches the directory structure in Google Drive
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/datasets/ACDC')
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/datasets/ACDC/training_output')
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATIONS_ZIP = DRIVE_DATA_ROOT / 'gt_detection' / 'gt_detection_trainval.zip'
IMAGES_ZIP = DRIVE_DATA_ROOT / 'rgb_anon' / 'rgb_anon_trainvaltest.zip'

for archive in [ANNOTATIONS_ZIP, IMAGES_ZIP]:
    if not archive.is_file():
        raise FileNotFoundError(
            f'Missing {archive}. Check your Google Drive directory structure.'
        )

print('Annotation archive:', ANNOTATIONS_ZIP)
print('Image archive:', IMAGES_ZIP)
print('Colab disk space:', shutil.disk_usage('/content').free // (1024 ** 3), 'GiB free')

# Colab working directories
ACDC_ROOT = Path('/content/datasets/ACDC')
PROCESSED_DIR = Path('/content/acdc_processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: Extract ACDC data to Colab disk (preserving directory structure)
import zipfile

CONDITIONS = {'fog', 'night', 'rain', 'snow'}
SPLITS = {'train', 'val'}

def safe_extract_member(archive, member, destination):
    target = (destination / member.filename).resolve()
    if not target.is_relative_to(destination.resolve()):
        raise ValueError(f'Unsafe path in dataset archive: {member.filename}')
    archive.extract(member, destination)

ACDC_ROOT.mkdir(parents=True, exist_ok=True)

# Extract annotation ZIP — all files, preserving structure
with zipfile.ZipFile(ANNOTATIONS_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise ValueError(f'Corrupt member in annotation archive: {bad_member}')
    annotation_members = [m for m in archive.infolist() if not m.is_dir()]
    missing_annotations = [
        m for m in annotation_members
        if not (ACDC_ROOT / m.filename).is_file()
    ]
    print(f'Extracting {len(missing_annotations)} annotation files...')
    for member in missing_annotations:
        safe_extract_member(archive, member, ACDC_ROOT)
    print(f'Annotations ready: {len(annotation_members)} files')

# Extract image ZIP — only adverse-condition train/val, preserving structure
with zipfile.ZipFile(IMAGES_ZIP) as archive:
    selected_members = []
    for member in archive.infolist():
        if member.is_dir():
            continue
        parts = Path(member.filename).parts
        if (
            'rgb_anon' in parts
            and CONDITIONS.intersection(parts)
            and SPLITS.intersection(parts)
        ):
            selected_members.append(member)
    if not selected_members:
        raise ValueError(
            'No RGB train/val files found in the image archive. '
            'Check that this is rgb_anon_trainvaltest.zip.'
        )
    missing_images = [
        m for m in selected_members
        if not (ACDC_ROOT / m.filename).is_file()
    ]
    print(f'Extracting {len(missing_images)} images...')
    for member in missing_images:
        safe_extract_member(archive, member, ACDC_ROOT)
    print(f'Images ready: {len(selected_members)} files')

# Verify structure
annotation_marker = 'instancesonly_fog_train_gt_detection.json'
if not (ACDC_ROOT / 'gt_detection' / 'fog' / annotation_marker).is_file():
    raise FileNotFoundError(
        f'Annotation extraction failed. Expected {ACDC_ROOT}/gt_detection/fog/{annotation_marker}'
    )
print(f'Data extracted to: {ACDC_ROOT}')

In [ ]:
# Cell 3: Convert COCO to YOLO format
result = subprocess.run(
    [
        sys.executable, 'dataset/run_conversion.py',
        '--acdc-root', str(ACDC_ROOT),
        '--output-dir', str(PROCESSED_DIR),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Conversion failed')

# Verify the output
print('Converted dataset structure:')
for split in ['train', 'val']:
    images = list((PROCESSED_DIR / 'images' / split).glob('*'))
    labels = list((PROCESSED_DIR / 'labels' / split).glob('*'))
    print(f'  {split}: {len(images)} images, {len(labels)} labels')

In [ ]:
# Cell 4: Train YOLO11n with efficient hyperparameters
import yaml

data_yaml = {
    'path': str(PROCESSED_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'person',
        1: 'rider',
        2: 'car',
        3: 'truck',
        4: 'bus',
        5: 'train',
        6: 'motorcycle',
        7: 'bicycle',
    },
}

yaml_path = PROCESSED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print(f'data.yaml written to {yaml_path}')

# Resume from last checkpoint if it exists on Drive
run_dir = DRIVE_CHECKPOINT_DIR / 'yolov11n_acdc_efficient'
last_checkpoint = run_dir / 'last.pt'
resume = last_checkpoint.is_file()

if resume:
    print(f'Resuming from {last_checkpoint}')
    model = YOLO(str(last_checkpoint))
else:
    print('Starting fresh training with yolo11n.pt')
    model = YOLO('yolo11n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=512,
    batch=32,
    patience=15,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    device=0,
    project=str(DRIVE_CHECKPOINT_DIR),
    name='yolov11n_acdc_efficient',
    exist_ok=True,
    plots=True,
    val=True,
    resume=resume,
)

# Copy checkpoints to Drive after training
for checkpoint in ['best.pt', 'last.pt']:
    src = run_dir / checkpoint
    if src.is_file():
        shutil.copy2(src, DRIVE_CHECKPOINT_DIR / checkpoint)
        print(f'Copied {checkpoint} to Drive')

print('Training complete!')
print(f'Best model: {DRIVE_CHECKPOINT_DIR / "best.pt"}')

In [ ]:
# Cell 5: Evaluate and compare with baseline
import json

eval_model = YOLO(str(DRIVE_CHECKPOINT_DIR / 'best.pt'))
eval_results = eval_model.val(
    data=str(yaml_path),
    imgsz=512,
    conf=0.001,
    plots=True,
    project=str(DRIVE_CHECKPOINT_DIR / 'evaluation'),
    exist_ok=True,
    device=0,
)

names = eval_results.names if isinstance(eval_results.names, dict) else {
    i: n for i, n in enumerate(eval_results.names)
}

summary = {
    'model': 'YOLO11n',
    'model_path': str(DRIVE_CHECKPOINT_DIR / 'best.pt'),
    'data': str(yaml_path),
    'imgsz': 512,
    'epochs': 50,
    'metrics': {
        'precision': round(float(eval_results.box.mp), 4),
        'recall': round(float(eval_results.box.mr), 4),
        'mAP@0.5': round(float(eval_results.box.map50), 4),
        'mAP@0.5:0.95': round(float(eval_results.box.map), 4),
    },
    'class_names': {str(k): v for k, v in names.items()},
}

metrics_file = DRIVE_CHECKPOINT_DIR / 'evaluation' / 'metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 60)
print('YOLO11n EVALUATION RESULTS')
print('=' * 60)
print(f"  Precision:     {summary['metrics']['precision']}")
print(f"  Recall:        {summary['metrics']['recall']}")
print(f"  mAP@0.5:       {summary['metrics']['mAP@0.5']}")
print(f"  mAP@0.5:0.95:  {summary['metrics']['mAP@0.5:0.95']}")
print()
print('Baseline (YOLOv8n):  mAP@0.5 = 0.2664, mAP@0.5:0.95 = 0.1538')
print(f"Improvement:         mAP@0.5 {'+' if summary['metrics']['mAP@0.5'] >= 0.2664 else ''}{summary['metrics']['mAP@0.5'] - 0.2664:.4f}, mAP@0.5:0.95 {'+' if summary['metrics']['mAP@0.5:0.95'] >= 0.1538 else ''}{summary['metrics']['mAP@0.5:0.95'] - 0.1538:.4f}")
print()
print(f'Metrics saved to: {metrics_file}')
print(f'Copy best.pt to your local models/ folder and update MODEL_PATH!')